In [ ]:
!pip install "rembg[gpu]" torch diffusers transformers accelerate opencv-python-headless pillow numpy requests


In [ ]:
import os
import io
import zipfile
import cv2
import torch
import numpy as np
import hashlib
import requests
import re
from PIL import Image
from rembg import remove
from diffusers import StableDiffusionImg2ImgPipeline
from transformers import pipeline as hf_pipeline
from google.colab import drive
from google.colab import files

print("Authenticating with Google Drive...")
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/PaperPlanes_Autopsy"
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Output directory secured at: {DRIVE_DIR}")

def get_file_hash(filepath):
    hasher = hashlib.md5()
    with open(filepath, 'rb') as f:
        buf = f.read()
        hasher.update(buf)
    return hasher.hexdigest()

def vivisect(image_path, layers=6):
    base_name = os.path.basename(image_path).split('.')[0]
    zip_path = os.path.join(DRIVE_DIR, f"{base_name}_strata.zip")
    
    if os.path.exists(zip_path):
        print(f"Artifact already exists: {zip_path}. Skipping.")
        return

    print(f"\n--- Vivisecting {image_path} ---")
    print("Phase 1: Stripping reality...")
    orig_pil = Image.open(image_path).convert("RGB")
    nobg_pil = remove(orig_pil)
    subject_mask = np.array(nobg_pil)[:, :, 3] > 0
    orig_cv = cv2.cvtColor(np.array(orig_pil), cv2.COLOR_RGB2BGR)

    print("Phase 2: Forging the hallucination...")
    sd_pipe = StableDiffusionImg2ImgPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16).to("cuda")
    sd_pipe.safety_checker = None
    
    prompt = "Raw, hyper-realistic photograph, incredibly detailed, 8k resolution, cinematic lighting, physical reality, sharp focus, real life"
    negative_prompt = "painting, illustration, drawing, art, canvas, brushstrokes, graffiti, wall, background, sketch, 2d, flat, texture, graphic, stylized"
    
    gen_pil = sd_pipe(prompt=prompt, negative_prompt=negative_prompt, image=orig_pil, strength=0.85, guidance_scale=12.0).images[0]
    gen_cv = cv2.cvtColor(np.array(gen_pil), cv2.COLOR_RGB2BGR)
    del sd_pipe
    torch.cuda.empty_cache()

    print("Phase 3: Bruteforce alignment...")
    gen_cv = cv2.resize(gen_cv, (orig_cv.shape[1], orig_cv.shape[0]))
    gray_src = cv2.cvtColor(orig_cv, cv2.COLOR_BGR2GRAY)
    gray_tgt = cv2.cvtColor(gen_cv, cv2.COLOR_BGR2GRAY)
    orb = cv2.ORB_create(MAX_FEATURES=5000)
    kp_src, des_src = orb.detectAndCompute(gray_src, None)
    kp_tgt, des_tgt = orb.detectAndCompute(gray_tgt, None)
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
    matches = bf.knnMatch(des_src, des_tgt, k=2)
    good_matches = [m for m, n in matches if m.distance < 0.75 * n.distance]

    if len(good_matches) > 10:
        src_pts = np.float32([kp_src[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
        tgt_pts = np.float32([kp_tgt[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)
        matrix, _ = cv2.findHomography(tgt_pts, src_pts, cv2.RANSAC, 5.0)
        aligned_cv = cv2.warpPerspective(gen_cv, matrix, (orig_cv.shape[1], orig_cv.shape[0])) if matrix is not None else gen_cv
    else:
        aligned_cv = gen_cv

    aligned_pil = Image.fromarray(cv2.cvtColor(aligned_cv, cv2.COLOR_BGR2RGB))

    print("Phase 4: Extracting depth map...")
    depth_pipe = hf_pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf", device=0)
    depth_array = np.array(depth_pipe(aligned_pil)["depth"]).astype(np.float32)
    del depth_pipe
    torch.cuda.empty_cache()

    print("Phase 5: Slicing strata...")
    subject_depth = depth_array[subject_mask]
    min_d, max_d = subject_depth.min(), subject_depth.max()
    normalized_depth = np.zeros_like(depth_array)
    if min_d != max_d:
        normalized_depth[subject_mask] = np.interp(depth_array[subject_mask], (min_d, max_d), (0, 255))

    bins = np.linspace(0, 255.1, layers + 1)
    orig_array = np.array(orig_pil)

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for i in range(layers):
            layer_mask = (normalized_depth >= bins[i]) & (normalized_depth < bins[i+1]) & subject_mask
            if np.any(layer_mask):
                layer_rgba = np.zeros((orig_array.shape[0], orig_array.shape[1], 4), dtype=np.uint8)
                layer_rgba[..., :3] = orig_array
                layer_rgba[..., 3] = cv2.GaussianBlur((layer_mask * 255).astype(np.uint8), (5, 5), 0)
                img_byte_arr = io.BytesIO()
                Image.fromarray(layer_rgba).save(img_byte_arr, format='PNG')
                zf.writestr(f"layer_{i:03d}.png", img_byte_arr.getvalue())

    print(f"Vivisection complete. Artifact permanently sealed in Drive: {zip_path}")

def raid_album(shared_url):
    print(f"Raiding the void: {shared_url}")
    response = requests.get(shared_url)
    
    matches = re.findall(r'(https:\/\/lh3\.googleusercontent\.com\/[a-zA-Z0-9\-_]+)', response.text)
    unique_urls = list(set(matches))
    
    image_urls = [url for url in unique_urls if len(url) > 60]
    
    if not image_urls:
        print("The mass grave is empty or fortified. Ensure you generated a 'Create link' share URL.")
        return []

    os.makedirs("victims", exist_ok=True)
    corpses = []
    
    for i, url in enumerate(image_urls):
        try:
            img_data = requests.get(url + "=d").content
            filename = f"victims/victim_{i:03d}.jpg"
            with open(filename, "wb") as f:
                f.write(img_data)
            corpses.append(filename)
            print(f"Exhumed: {filename}")
        except Exception as e:
            print(f"Failed to exhume {url}: {e}")
            
    return corpses

# ---------------------------------------------------------
# EXECUTION
# ---------------------------------------------------------
ALBUM_URL = "" # Paste your Google Photos shared album URL here. Leave blank for manual upload.

processed_hashes = set()
valid_extensions = {'.png', '.jpg', '.jpeg', '.webp'}

if ALBUM_URL:
    print("Album URL detected. Initiating raid...")
    victims = raid_album(ALBUM_URL)
    for victim in victims:
        file_hash = get_file_hash(victim)
        if file_hash in processed_hashes:
            print(f"Skipping duplicate file from album.")
            continue
        processed_hashes.add(file_hash)
        vivisect(victim, layers=6)
else:
    print("No Album URL provided. Initiating manual upload widget...")
    uploaded = files.upload()
    for filename in uploaded.keys():
        ext = os.path.splitext(filename)[1].lower()
        if ext not in valid_extensions:
            print(f"Skipping non-image file: {filename}")
            continue
            
        file_hash = get_file_hash(filename)
        if file_hash in processed_hashes:
            print(f"Skipping duplicate file: {filename}")
            continue
            
        processed_hashes.add(file_hash)
        vivisect(filename, layers=6)
